In [1]:
from gensim.models import KeyedVectors

model_path = "model.bin"

model = KeyedVectors.load_word2vec_format(model_path, binary=True)

In [14]:
import re
import simplemma

def lemmatize(word):
    return simplemma.lemmatize(word, lang="pl")

def group_similar_words_word2vec(text, threshold=0.7):

    words = re.findall(r'\b[a-zA-Ząćęłńóśźż]+\b', text.lower())
    # words_lem = [lemmatize(w) for w in words]
    words_in_text = set(words)

    grouped = {}
    seen = set()

    for w in words_in_text:
        if len(w) < 3:
            continue
        if w in seen or w not in model:
            continue

        similar = [(word, score) for word, score in model.most_similar(w, topn=50)
                   if word in words_in_text and score > threshold]

        if similar:
            group = {word for word, _ in similar}
            group.discard(w)
            if group:
                grouped[w] = sorted(group)
                seen.update(group)
                seen.add(w)

    return grouped


In [15]:
with open("tekst.txt", "r", encoding="utf-8") as f:
    tekst = f.read()

wynik = group_similar_words_word2vec(tekst)

with open("results.txt", "w", encoding="utf-8") as out:
    for rep, group in wynik.items():
        out.write(f"{rep}: {group}\n")